# GISLR — ASL-LEX rule-based phonology engine: no pose vs. with pose

**What this notebook does.** Diagnostic / research notebook (not a training pipeline stage — nothing here
fits a model). Part of **recognition_2** (TODO §14), following the critique in
`docs/reports/phonology-pipeline-proposal.md`. Answers two questions: **can a rule-based engine identify
ASL-LEX 2.0's 18 phonological parameters from a GISLR clip well enough to match ASL-LEX's own coding of that
clip's gloss?**, and **does adding MediaPipe pose add a significant benefit over hands + face alone, or is
discarding pose feasible?** (user ask, 2026-09-27).

Uses `sb.recognize.recognition2.rules` (the engine — new, isolated, never edits `sb.recognize.phonology`
or `aslex`, only imports their pose-independent geometry helpers) and `sb.recognize.recognition2.validate`
(scoring against ASL-LEX ground truth). Restricted to the **233/250 GISLR glosses that map into ASL-LEX**
(the gloss-mapping check earlier in this conversation, 2026-09-26/27) — ASL-LEX is the only ground truth
available, so nothing here can be scored for the other 17.

**Two variants, same thresholds, same everything else** (`rules.extract(clip, use_pose=...)`):

- **`no_pose`**: hands + face only. Anchor = face centroid, scale = inter-eye distance.
- **`with_pose`**: hands and face stay the *primary* signal for every hand-internal parameter (user:
  "prioritize face and hands") — pose only fills the gaps `no_pose` cannot reach: anchor becomes
  mid-shoulder + shoulder-width scale (matching `phonology.py`), `major_location` additionally attempts
  `Body`/`Arm`, and `movement`/`repeated_movement` fall back to the pose wrist's trajectory when the hand
  itself is undetected for more than half the nucleus.

**Retuned this run** (found analyzing the first `no_pose`-only run, TODO §14, 2026-09-27): a real bug where
`major_location`'s "Hand" check used a median instead of a min (a brief touch was washed out — fixed in
`rules.py`), plus three threshold recalibrations in `stage1-rules.json` (`spread_t`, `amp_t`/`apt_amp_t`,
`flat_base_t`) — see that file's `_comment` for the evidence.

**Artifacts:**

| path | what |
|---|---|
| `data/cache/gislr_aslex_rules/agreement_no_pose.csv` | 18-parameter agreement table, hands + face only |
| `data/cache/gislr_aslex_rules/agreement_with_pose.csv` | 18-parameter agreement table, with pose fallbacks |
| `data/cache/gislr_aslex_rules/agreement_comparison.csv` | side by side, with the delta |
| `data/cache/gislr_aslex_rules/records_no_pose.json` / `records_with_pose.json` | every scored clip's truth/prediction, per variant |

**Resumability**: fast enough (rule evaluation, no training) to just re-run in full; no manifest needed.


## Setup

In [1]:
# ============================================================
# Imports, tunables, resolved paths
# ============================================================
import json
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sb.core.paths import CACHE_DIR, EXPERIMENTS_DIR, gislr_dir
from sb.core.vocab import load_label_map
from sb.recognize.aslex import gloss_entries, signdata
from sb.recognize.recognition2 import rules, validate

CONFIG_PATH = EXPERIMENTS_DIR / "recognition2" / "configs" / "stage1-rules.json"
THRESHOLDS = rules.load_thresholds(CONFIG_PATH)

MAX_CLIPS_PER_GLOSS = 6  # tunable: raise for a fuller run once the sample-size result looks sane
VARIANTS = [("no_pose", False), ("with_pose", True)]

OUT_DIR = CACHE_DIR / "gislr_aslex_rules"
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = gislr_dir()

print(f"data:    {DATA_DIR}")
print(f"config:  {CONFIG_PATH.name}")
print(f"output:  {OUT_DIR}")
print(f"max clips/gloss this run: {MAX_CLIPS_PER_GLOSS}")
print(f"variants: {[v for v, _ in VARIANTS]}")


data:    C:\Users\user2\.cache\kagglehub\datasets\bracu23101281\gislr-stratified\versions\1
config:  stage1-rules.json
output:  C:\Users\user2\repository\data\cache\gislr_aslex_rules
max clips/gloss this run: 6
variants: ['no_pose', 'with_pose']


## 1. ASL-LEX-mapped glosses and the handshape lookup table

`rules.build_handshape_lookup` reads the **entire** ASL-LEX lexicon (not just the 233 mapped glosses) to
build `{(selected_fingers, flexion, spread, thumb_position, thumb_contact): modal Handshape}` —
Handshape is *defined* by signdataKEY.csv as exactly this combination, so this is a lookup of ASL-LEX's own
coding scheme, not a fit to the glosses this notebook scores against. Shared by both variants — it does
not depend on pose.

In [2]:
# ============================================================
# ASL-LEX-mapped glosses + handshape lookup
# ============================================================
label_map = load_label_map(DATA_DIR)
glosses = sorted(label_map.keys())
ENTRIES = gloss_entries(glosses)
MAPPED = [g for g in glosses if g in ENTRIES]
SIGNDATA = signdata().set_index("EntryID")
LOOKUP, IS_MARKED = rules.build_handshape_lookup()

print(f"GISLR glosses: {len(glosses)}, ASL-LEX-mapped: {len(MAPPED)}")
print(f"handshape lookup: {len(LOOKUP)} sub-feature combinations -> {len(IS_MARKED)} handshapes")


GISLR glosses: 250, ASL-LEX-mapped: 233
handshape lookup: 61 sub-feature combinations -> 35 handshapes


## 2. Score both variants against ASL-LEX ground truth

For each mapped gloss: look up its ASL-LEX ground truth per parameter (`validate.ground_truth_for_gloss`,
shared across variants), run the rule engine on the **same** up-to-`MAX_CLIPS_PER_GLOSS` clips for both
`use_pose` settings (same seed, so both variants see identical clips — any difference in the tables is the
pose ablation, not a different sample), and record both.

In [3]:
# ============================================================
# Run the engine over every mapped gloss's clips, both variants
# ============================================================
train = pd.read_csv(DATA_DIR / "train.csv")
rng = np.random.default_rng(0)
clip_rows_by_gloss = {}
for g in MAPPED:
    rows = train[train["sign"] == g]
    if len(rows) > MAX_CLIPS_PER_GLOSS:
        rows = rows.sample(n=MAX_CLIPS_PER_GLOSS, random_state=rng.integers(1 << 31))
    clip_rows_by_gloss[g] = rows

RECORDS = {}
for variant, use_pose in VARIANTS:
    recs = []
    for g in tqdm(MAPPED, desc=f"glosses ({variant})"):
        truth = validate.ground_truth_for_gloss(g, ENTRIES, SIGNDATA)
        for _, row in clip_rows_by_gloss[g].iterrows():
            clip = rules.load_clip(DATA_DIR / row["npz_relpath"])
            feats = rules.extract(clip, use_pose=use_pose)
            pred = rules.codes(feats, THRESHOLDS, LOOKUP, IS_MARKED)
            recs.append({"gloss": g, "uid": row["uid"], "truth": truth, "pred": pred})
    RECORDS[variant] = recs
    (OUT_DIR / f"records_{variant}.json").write_text(json.dumps(recs, indent=1), encoding="utf-8")
    print(f"{variant}: scored {len(recs)} clips over {len(MAPPED)} glosses")


glosses (no_pose):   0%|          | 0/233 [00:00<?, ?it/s]

no_pose: scored 1398 clips over 233 glosses


glosses (with_pose):   0%|          | 0/233 [00:00<?, ?it/s]

with_pose: scored 1398 clips over 233 glosses


## 3. Per-parameter agreement, both variants side by side

Three numbers per parameter per variant (`validate.agreement_table`): **coverage** (of clips with ASL-LEX
ground truth, how many the engine could even attempt), **agreement** (among attempted clips, how often it
matches ASL-LEX), **chance** (1 / category count). The `with_pose` − `no_pose` delta columns are the direct
answer to "does pose add a significant benefit": positive on `major_location` (new `Body`/`Arm` reach) is
expected by construction; a positive delta anywhere else (`movement`/`repeated_movement`, via the pose-wrist
fallback) would be the more interesting finding, since those parameters aren't pose-gated by definition.

In [4]:
# ============================================================
# Agreement tables + comparison
# ============================================================
TABLES = {}
for variant, _ in VARIANTS:
    tbl = validate.agreement_table(RECORDS[variant]).set_index("parameter")
    tbl.to_csv(OUT_DIR / f"agreement_{variant}.csv")
    TABLES[variant] = tbl

cmp = TABLES["no_pose"][["coverage", "agreement", "chance"]].add_suffix("_no_pose").join(
    TABLES["with_pose"][["coverage", "agreement"]].add_suffix("_with_pose"))
cmp["coverage_delta"] = cmp["coverage_with_pose"] - cmp["coverage_no_pose"]
cmp["agreement_delta"] = cmp["agreement_with_pose"] - cmp["agreement_no_pose"]
cmp = cmp.sort_values("agreement_delta", ascending=False)
cmp.to_csv(OUT_DIR / "agreement_comparison.csv")
cmp


,coverage_no_pose,agreement_no_pose,chance_no_pose,coverage_with_pose,agreement_with_pose,coverage_delta,agreement_delta
parameter,,,,,,,
repeated_movement,1.000000,0.531674,0.500000,1.000000,0.532428,0.0,0.000754
sign_type,1.000000,0.659591,0.200000,1.000000,0.659591,0.0,0.000000
handshape,0.386097,0.123644,0.025641,0.386097,0.123644,0.0,0.000000
selected_fingers,1.000000,0.468599,0.111111,1.000000,0.468599,0.0,0.000000
marked_handshape,0.384434,0.564417,0.500000,0.384434,0.564417,0.0,0.000000
flexion_change,1.000000,0.661859,0.500000,1.000000,0.661859,0.0,0.000000
spread,0.839744,0.554198,0.500000,0.839744,0.554198,0.0,0.000000
spread_change,1.000000,0.764493,0.500000,1.000000,0.764493,0.0,0.000000
flexion,0.841065,0.273749,0.166667,0.841065,0.273749,0.0,0.000000


## 4. Reading the comparison

- **`coverage_delta` > 0 on `major_location`/`minor_location`** is expected — pose unlocks `Body`/`Arm`,
  which were structurally `NOT_COMPUTABLE` in `no_pose`. The question is whether `agreement_delta` on those
  rows is also positive (pose helping) or flat/negative (the new `Body`/`Arm` thresholds firing on the
  wrong clips — they are brand-new and unlike every other threshold here were not calibrated against a
  full run before this notebook executed).
- **`agreement_delta` on `movement`/`repeated_movement`** isolates the pose-wrist fallback specifically
  (these parameters are not pose-gated by definition — any change here comes only from clips where the
  hand itself dropped out of the nucleus for more than half its span).
- **Every hand-internal parameter** (`handshape`, `selected_fingers`, `flexion`*, `spread`*, `thumb_*`,
  `ulnar_rotation`, `non_dominant_handshape`) should show `agreement_delta` ≈ 0 — pose does not touch their
  inputs at all (per "prioritize face and hands"). A large delta there would mean the anchor/scale change
  (face-centroid+eye-distance vs. mid-shoulder+shoulder-width) is doing more than expected, since that is
  the one thing that changes for every parameter, not just the pose-gated ones.
